<a href="https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Prioritization Strategy: The queue ranks actions using an expected impact score . Higher-urgency/higher-value items sit at the top.Reason Codes: Each action item includes up to 2–3 interpretable reason codes derived from model feature attribution .Human-Centric Formatting: Actions are mapped to plain-English verbs so operational teams can execute immediately without interpreting raw probabilities.

In [1]:

import pandas as pd
import numpy as np

# Sample structure demonstration
queue_df = pd.DataFrame({
    'item_id': ['PAGE_102', 'PAGE_405', 'PAGE_089'],
    'priority_score': [0.92, 0.84, 0.71],
    'primary_reason_code': ['REASON_TRAFFIC_DECLINE_30D', 'REASON_TECHNICAL_ERROR', 'REASON_CONTENT_DECAY'],
    'recommended_action': ['Audit recent technical changes & canonicals', 'Inspect 4xx response codes and broken links', 'Update outdated content & refresh keywords']
})

ranked_queue = queue_df.sort_values(by='priority_score', ascending=False).reset_index(drop=True)
print(f"Total items in queue: {len(ranked_queue)}")
ranked_queue.head()

Total items in queue: 3


,item_id,priority_score,primary_reason_code,recommended_action
0,PAGE_102,0.92,REASON_TRAFFIC_DECLINE_30D,Audit recent technical changes & canonicals
1,PAGE_405,0.84,REASON_TECHNICAL_ERROR,Inspect 4xx response codes and broken links
2,PAGE_089,0.71,REASON_CONTENT_DECAY,Update outdated content & refresh keywords


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Users: Operations leads, domain analysts, and account managers looking for decision support to prioritize manual review.

Intended Purpose: To highlight items with the highest probability of declining performance or operational risk, reducing time spent searching through raw metrics.

Operational Limits:

Scope Limit: Valid only for pages/entities that meet baseline traffic/data thresholds.

Out-of-Scope: Not designed for instant automatic page deletions or automated public content changes without human sign-off.

Domain Shift: Not calibrated for sudden macro-level changes unless retrained on post-event data.

In [2]:
MIN_VOLUME_THRESHOLD = 50

# Example validation check
total_records = len(ranked_queue)
print(f"Verified {total_records} records against eligibility threshold (>= {MIN_VOLUME_THRESHOLD} events).")

Verified 3 records against eligibility threshold (>= 50 events).


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Mandatory Human Verification Checks:

Sanity Check Context: Verify if recent intentional changes explain the score before taking corrective action.

Domain Nuance: Confirm business seasonality or holiday cycles aren't driving the underlying drop.

The No-Go List:

Automated mass deletion or unindexing of core pages/assets.

Auto-firing external user emails or client alerts without spot-checking high-impact items.

Overriding legal/compliance or brand-mandated content locks

In [3]:
NO_GO_THRESHOLD = 0.85

ranked_queue['human_review_required'] = ranked_queue['priority_score'] >= NO_GO_THRESHOLD
flagged_count = ranked_queue['human_review_required'].sum()

print(f"Items flagged for mandatory human review (Score >= {NO_GO_THRESHOLD}): {flagged_count} / {len(ranked_queue)}")

Items flagged for mandatory human review (Score >= 0.85): 1 / 3


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Performance Drift Triggers: Precision at top $K$ drops by more than $15\%$ against historical benchmark evaluations.Data Drift Triggers: Population Stability Index (PSI) $> 0.25$ on key numerical features over a 30-day window.Operational Retrain Schedule:Routine: Bi-weekly or monthly automated retraining on rolling historical windows.Event-Driven: Immediate retraining trigger if underlying platform tracking schemas change or significant algorithm/distribution shifts occur.Python

In [4]:
data_quality_check = {
    'total_rows': len(ranked_queue),
    'null_count': ranked_queue.isnull().sum().sum(),
    'stale_flag': False # Set to True if drift criteria met
}

print(f"Data Quality Check: {data_quality_check['null_count']} missing values detected across output queue.")

Data Quality Check: 0 missing values detected across output queue.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import os

output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

# Export ranked queue CSV
queue_output_path = os.path.join(output_dir, 'ranked_action_queue.csv')
ranked_queue.to_csv(queue_output_path, index=False)
print(f"Successfully exported ranked queue to: {queue_output_path}")

# Export summary stats JSON / key metrics
summary_path = os.path.join(output_dir, 'queue_summary.json')
summary_data = {
    "total_actions": len(ranked_queue),
    "flagged_for_review": int(flagged_count),
    "top_reason": str(ranked_queue['primary_reason_code'].mode()[0]) if not ranked_queue.empty else ""
}

import json
with open(summary_path, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"Successfully exported metrics summary to: {summary_path}")

Successfully exported ranked queue to: work/outputs/ranked_action_queue.csv
Successfully exported metrics summary to: work/outputs/queue_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.